# SER fallback examples for UK Biobank blood traits

These examples illustrate the two SER fallback criteria in Extended Data Figure 1. Set `ld_ref` in the submit cells to use colocalization fits with 1000 Genomes EUR LD (`19`, `UKB_coloclization_1000_genome`; n = 489, `R_finite = 489`) or ADSP sketch LD (`18`, `UKB_coloclization`). We use 1000 Genomes LD for these examples because the ADSP sketch yielded few cases well above the Q_art threshold. Of 152 flagged ADSP fits, Q_art flagged 17, including 7 by Q_art alone and 10 by both criteria. SuSiE-rRSS sets `R_reliability_flag` when
- **Q_art** > 0.1 (artifact threshold): the fraction of the fitted z-scale residual that lies in low-eigenvalue directions of the reference LD, Q_art = ‖r_low‖² / ‖r‖², or
- **D_CS,max** ≥ 30 (sensitivity threshold): for some credible set, the largest log-BF attenuation of a variant in it, nominal log BF (s.e. not inflated) − adjusted log BF (s.e. inflated for R uncertainty), i.e. the log of the BF ratio.

Three steps:
1. `[fallback_cases_select]` (interactive, seconds): reads the per-record coloc results (`18` `[coloc]`), classifies each flagged SuSiE-rRSS fit as **Q_art only**, **D_CS only** or **both** with the stored Q_art and DCS_max (as `FigED1CD.SER_Fallback_Threshold`), and draws candidates per class (default 12 / 12 / 8, one per region). Eligible: SuSiE-RSS has at least 2 credible sets (so the SER fallback visibly changes the result), Q_art ≥ 0.15 for the Q_art classes and D_CS,max ≥ 60 for the D_CS classes (away from the thresholds); the largest regions (bp) first. Select more candidates than the 8 / 8 / 5 shown because the current susieR build can assign a different fallback class on refitting.
2. `[ukb_case_inputs]` of `31.UKB_Power_Cases`, with `--units` from step 1 and `--ld_ref`: rebuilds the inputs exactly as `UKB_fine.R` did (`18` or `19`).
3. `[fallback_case_finemap]` (one task per unit × method): SuSiE-RSS, SuSiE-rRSS and in-sample SuSiE-RSS, the same calls as `31` `[ukb_case_finemap]`. For SuSiE-rRSS save diagnostics that `cleanup_model` normally drops. These include the L × p log-BF attenuation matrix and the per-variant s.e.² inflation of the last SER update, and the residual r and its low-eigenvalue part r_low behind the final Q_art, with the squared projection of r on every eigenvector of the reference LD and its eigenvalue (where in the LD spectrum the residual lies).

Output: `/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_SER_fallback_cases_1000G/` (1000 Genomes) or `UKB_SER_fallback_cases/` (ADSP)
- `units/fallback_cases.tsv`: every flagged pair with its class and whether it is a candidate; `units/fallback_cases_units.txt`: the candidates;
- `result/<unit>/` (`<unit>.inputs.rds`, `<unit>.<method>.rds`); `<unit>` = `<chr>_<start>_<end>.<trait>`.

Plots: `Data_Processing/Figure_5/UKB_SER_Fallback_Cases` → `Figures/Figure_5/UKB_SER_Fallback_Cases`.

## 1. Select cases

In [ ]:
[fallback_cases_select]
# Classify the flagged SuSiE-rRSS fits by trigger and draw candidates per class.
parameter: coloc_dir = path("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_coloclization/coloc/result")
parameter: repo = path("/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper")
parameter: n_q = 12
parameter: n_dcs = 12
parameter: n_both = 8
# candidates per class: Q_art only, D_CS only, both
parameter: artifact_threshold = 0.1
parameter: sensitivity_threshold = 30.0
# eligible candidates, away from the thresholds: Q_art >= q_min (Q_art only, both), D_CS,max >= dcs_min (D_CS only, both)
parameter: q_min = 0.15
parameter: dcs_min = 60.0
# and SuSiE-RSS with at least min_rss_cs credible sets (not a single-effect region)
parameter: min_rss_cs = 2
parameter: cwd = path("output")
parameter: container = ""
output: f'{cwd:a}/fallback_cases.tsv', f'{cwd:a}/fallback_cases_units.txt'
R: expand = '${ }', stdout = f"{_output[0]:n}.stdout", stderr = f"{_output[0]:n}.stderr", container = container
    source(file.path(${repo:r}, "R", "utils.R"))          # read_coloc_records
    art <- ${artifact_threshold}; sens <- ${sensitivity_threshold}

    recs <- read_coloc_records(${coloc_dir:r})
    rss  <- recs %>% filter(method == "susie_rss_res") %>% distinct(gene, tissue, .keep_all = TRUE) %>%
      select(gene, tissue, n_cs_rss = n_outsample_cs)

    ## gene = region as named by 18 (res_<region>.rds), tissue = trait
    fb <- recs %>% filter(method == "susie_eb_mix_res") %>% distinct(gene, tissue, .keep_all = TRUE) %>%
      mutate(q_trig = is.finite(Q_art) & Q_art > art,
             d_trig = is.finite(DCS_max) & DCS_max >= sens) %>%
      filter(q_trig | d_trig) %>%
      mutate(trigger = factor(case_when(q_trig & d_trig ~ "Both", q_trig ~ "Q_art only", TRUE ~ "D_CS only"),
                              levels = c("Q_art only", "D_CS only", "Both")),
             unit = paste0(str_replace_all(gene, "[:-]", "_"), ".", tissue)) %>%
      left_join(rss, by = c("gene", "tissue")) %>%
      mutate(n_cs_rss = coalesce(as.integer(n_cs_rss), 0L)) %>%
      select(unit, region = gene, trait = tissue, trigger, Q_art, DCS_max, reliability_flag,
             n_cs_rrss = n_outsample_cs, n_cs_rss, n_cs_insample = n_insample_cs)

    cat(sprintf("flagged SuSiE-rRSS fits: %d (stored flag TRUE: %d)\n", nrow(fb), sum(fb$reliability_flag %in% TRUE)))
    ## eligible: away from the thresholds, and SuSiE-RSS not a single-effect fit
    fb <- fb %>%
      mutate(region_kb = (as.numeric(str_match(region, "[:_]([0-9]+)[-_]([0-9]+)$")[, 3]) -
                          as.numeric(str_match(region, "[:_]([0-9]+)[-_]([0-9]+)$")[, 2])) / 1e3,
             eligible = n_cs_rss >= ${min_rss_cs} &
                        (trigger == "D_CS only" | Q_art >= ${q_min}) &
                        (trigger == "Q_art only" | DCS_max >= ${dcs_min}))
    fb %>% group_by(trigger) %>%
      summarise(n = n(), regions = n_distinct(region), rss_multi_cs = sum(n_cs_rss >= ${min_rss_cs}),
                eligible = sum(eligible), eligible_regions = n_distinct(region[eligible]), .groups = "drop") %>%
      as.data.frame() %>% print()

    ## candidates: eligible, one per region, largest regions first
    n_target <- c("Q_art only" = ${n_q}, "D_CS only" = ${n_dcs}, "Both" = ${n_both})
    cand <- fb %>% filter(eligible) %>%
      arrange(trigger, desc(region_kb), desc(n_cs_rss)) %>%
      group_by(trigger) %>% distinct(region, .keep_all = TRUE) %>%
      mutate(rank = row_number()) %>% filter(rank <= n_target[as.character(trigger[1])]) %>%
      ungroup() %>% select(unit, rank)

    fb <- fb %>% left_join(cand, by = "unit") %>% mutate(candidate = !is.na(rank)) %>%
      arrange(trigger, rank)
    print(as.data.frame(filter(fb, candidate)))
    write_tsv(fb, ${_output[0]:r})
    writeLines(fb$unit[fb$candidate], ${_output[1]:r})

In [ ]:
# run interactively on a login node (seconds)
ld_ref="1000G"     # 1000G (19, UKB_coloclization_1000_genome) or ADSP (18, UKB_coloclization)
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output"
if [ "${ld_ref}" = "1000G" ]; then
  job_name="UKB_SER_fallback_cases_1000G"; coloc_dir="${base_dir}/UKB_coloclization_1000_genome/coloc/result"
else
  job_name="UKB_SER_fallback_cases";       coloc_dir="${base_dir}/UKB_coloclization/coloc/result"
fi
units_dir="${base_dir}/${job_name}/units"
mkdir -p ${units_dir}
sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/34.UKB_SER_Fallback_Cases.ipynb" fallback_cases_select \
    --coloc_dir ${coloc_dir} --cwd ${units_dir}
wc -l ${units_dir}/fallback_cases_units.txt

## 2. Fine-mapping

Inputs come from `31` `[ukb_case_inputs]` (submit cell below). Same fits as `31` `[ukb_case_finemap]`; for SuSiE-rRSS the run also saves
- `full_fit$keep_R_bf_attenuation` (L × p) and `full_fit$keep_shat2_inflation` (p): `cleanup_model.default` is wrapped to keep them;
- `qart`: the residual `r_fit` from the last `compute_Q_art` call, which gives the reported Q_art, and `r_low`, its projection on the eigenvectors of the reference LD with eigenvalue ≤ δ (`qart_matches_fit` checks ‖r_low‖² / ‖r‖² = Q_art); `qart_trace`: Q_art, ‖r_low‖², ‖r‖² at every call.

In [ ]:
[fallback_case_finemap]
# One task per (region x trait, method), in a fresh R session that reads the 31 [ukb_case_inputs] output.
parameter: units = list
# <chr>_<start>_<end>.<trait> (fallback_cases_units.txt)
parameter: methods = ["susie_rss", "susie_rrss", "insample"]
parameter: cwd = path("output")
parameter: job_size = 1
parameter: walltime = "20h"
parameter: mem = "60G"
parameter: numThreads = 1
parameter: container = ""
input: for_each = ["units", "methods"]
output: f'{cwd:a}/{_units}/{_units}.{_methods}.rds'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime, mem = mem, cores = numThreads,
      tags = f'{step_name}_{_units}_{_methods}'
R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container
    library(tidyverse)
    library(MASS)
    library(crayon)                 # used by the sourced susieR (combine_styles)
    library(Rfast)

    # susieR GitHub version (SuSiE-RSS / SuSiE-rRSS with eb_mix)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                         full.names = TRUE, pattern = "\\.R$")) source(f)
    select <- dplyr::select

    ## ---- keep the diagnostics that cleanup_model strips, and the residual behind Q_art ----
    stopifnot(exists("cleanup_model.default"), exists("compute_Q_art"))
    .cleanup_orig <- cleanup_model.default
    cleanup_model.default <- function(data, params, model, ...) {
      infl <- model$shat2_inflation; atten <- model$R_bf_attenuation
      model <- .cleanup_orig(data, params, model, ...)
      model$keep_shat2_inflation  <- infl      # per-variant s.e.^2 inflation of the last SER update
      model$keep_R_bf_attenuation <- atten     # L x p: nominal - adjusted log BF, last update of each effect
      model
    }
    .qart_trace <- list(); .qart_last <- NULL
    .compute_Q_art_orig <- compute_Q_art
    compute_Q_art <- function(eigen_R, r_fit, eig_delta_rel = 1e-3, eig_delta_abs = 0, residual_norm_floor = 1e-12) {
      out <- .compute_Q_art_orig(eigen_R, r_fit, eig_delta_rel, eig_delta_abs, residual_norm_floor)
      den <- sum(r_fit^2)
      .qart_last <<- list(Q_art = out$Q_art, evaluable = out$evaluable, r_fit = as.numeric(r_fit),
                          low_eigen_count = out$low_eigen_count, eig_delta = out$eig_delta, eigen_R = eigen_R)
      .qart_trace[[length(.qart_trace) + 1]] <<- tibble(call = length(.qart_trace) + 1, Q_art = out$Q_art,
                                                        num = out$Q_art * den, den = den,
                                                        low_eigen_count = out$low_eigen_count)
      out
    }
    ## r_low: projection of the last residual on the eigenvectors with eigenvalue <= delta (once, after the fit)
    qart_finish <- function(q) {
      if (is.null(q)) return(NULL)
      low <- which(q$eigen_R$values <= q$eig_delta)
      V   <- q$eigen_R$vectors[, low, drop = FALSE]
      q$r_low   <- if (length(low)) as.numeric(V %*% crossprod(V, q$r_fit)) else rep(0, length(q$r_fit))
      ## every eigen-direction: eigenvalue and squared projection of r (sum over eigenvalue <= delta = Q_art numerator)
      q$eig_values <- as.numeric(q$eigen_R$values)
      q$eig_proj2  <- as.numeric(crossprod(q$eigen_R$vectors, q$r_fit))^2
      q$eig_max <- max(q$eigen_R$values); q$p <- length(q$r_fit)
      q$eigen_R <- NULL
      q                                              # Q_art = sum(r_low^2) / sum(r_fit^2)
    }

    method <- "${_methods}"
    inp    <- readRDS(file.path(${_output:dr}, "${_units}.inputs.rds"))
    cat("unit:", inp$unit, " method:", method, " p =", length(inp$z_sketch), "\n")

    ## reference LD as the original fit used it: X = sketch (18, ADSP) or R = sketch_LD (19, 1000G)
    ld_arg <- if (identical(inp$ld_input, "R")) list(R = inp$sketch_LD) else list(X = inp$sketch)
    rss_ref <- function(...) do.call(susie_rss, c(list(z = inp$z_sketch, n = inp$n), ld_arg, list(...)))

    t0  <- Sys.time()
    out <- tryCatch(switch(method,
      susie_rss = list(fit = rss_ref()),
      susie_rrss = {
        .qart_trace <- list(); .qart_last <- NULL
        fit  <- rss_ref(R_finite = inp$B_sketch, R_mismatch = "eb_mix")
        d    <- fit$R_finite_diagnostics
        flag <- isTRUE(d$R_reliability_flag)
        if (flag && is.null(d$ser_model)) stop("R_reliability_flag set but no ser_model in the fit")
        qart <- qart_finish(.qart_last); rm(.qart_last)
        qart_ok <- !is.null(qart) && isTRUE(all.equal(qart$Q_art, d$Q_art)) &&
                   (!qart$evaluable || isTRUE(all.equal(sum(qart$r_low^2) / sum(qart$r_fit^2), d$Q_art)))
        list(fit = if (flag) d$ser_model else fit,                        # pipeline result
             full_fit = fit, ser_fallback = flag,                         # fit before fallback, flag
             artifact_flag = isTRUE(d$artifact_flag), sensitivity_flag = isTRUE(d$R_sensitivity_flag),
             Q_art = d$Q_art, cs_max = d$bf_attenuation$cs_max,
             qart = qart, qart_matches_fit = qart_ok, qart_trace = bind_rows(.qart_trace))
      },
      insample = list(fit = susie_rss(z = inp$insample_z, R = inp$insample_LD, n = inp$n)),
      stop("unknown method: ", method)),
      error = function(e) list(error = conditionMessage(e)))
    out$time_s <- as.numeric(difftime(Sys.time(), t0, units = "secs"))
    out$unit <- inp$unit; out$region <- inp$region; out$trait <- inp$trait; out$method <- method
    out$variant_ids <- inp$variant_ids
    saveRDS(out, ${_output:r})
    if (!is.null(out$error)) cat("[FAIL]", out$error, "\n") else
      cat(sprintf("%s %s: %d CS, %.1f s%s\n", inp$unit, method, length(out$fit$sets$cs), out$time_s,
                  if (isTRUE(out$ser_fallback))
                    sprintf(" (SER fallback: Q_art %.3f, D_CS,max %.1f)", out$Q_art, max(c(0, out$cs_max), na.rm = TRUE)) else ""))

Submit: inputs (`31` `[ukb_case_inputs]`), then the three fits, in one job.

In [ ]:
# ===== Base config =====
ld_ref="1000G"     # as in the select cell: 1000G or ADSP
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/"
if [ "${ld_ref}" = "1000G" ]; then
  job_name="UKB_SER_fallback_cases_1000G"; res_dir="${base_dir}/UKB_coloclization_1000_genome/result"
else
  job_name="UKB_SER_fallback_cases";       res_dir="${base_dir}/UKB_coloclization/result"
fi
job_dir="${base_dir}/${job_name}"
units_file="${job_dir}/units/fallback_cases_units.txt"      # from [fallback_cases_select]

mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 4
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=60000]
#BSUB -W 72:00
#BSUB -o WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin
units=$(cat UNITS_FILE)
sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/31.UKB_Power_Cases.ipynb" ukb_case_inputs       --units ${units} --ld_ref LD_REF --res_dir RES_DIR --cwd WORK_DIR/result --mem 60G -J 4 && \
sos run "/sc/arion/projects/ad-omics/haochen/rss_qc/susie-rRSS-paper/simulation_code/2026_Simulation_Sinai/34.UKB_SER_Fallback_Cases.ipynb" fallback_case_finemap --units ${units} --cwd WORK_DIR/result --mem 60G -J 4 -s force
EOF

sed -e "s|WORK_DIR|${job_dir}|g" -e "s|LD_REF|${ld_ref}|g" -e "s|RES_DIR|${res_dir}|g" -e "s|JOB_NAME|${job_name}|g" -e "s|UNITS_FILE|${units_file}|g" ${job_name}.sh > ${job_name}.sh.submit
sb ${job_name}.sh.submit

## 3. Check

Credible sets per unit and method, the stored and refitted SuSiE-rRSS diagnostics, and run time.

In [1]:
library(tidyverse)
`%||%` <- function(a, b) if (is.null(a)) b else a
root  <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/UKB_SER_fallback_cases_1000G"   # or UKB_SER_fallback_cases (ADSP)
cases <- read_tsv(file.path(root, "units", "fallback_cases.tsv"), show_col_types = FALSE) %>% filter(candidate)

fits <- list.files(file.path(root, "result"), pattern = "\\.(susie_rss|susie_rrss|insample)\\.rds$",
                   recursive = TRUE, full.names = TRUE)
chk <- map_dfr(fits, function(f) {
  o <- readRDS(f)
  tibble(unit = o$unit, method = o$method, error = o$error %||% NA_character_,
         n_cs = length(o$fit$sets$cs),
         n_cs_before_fallback = if (!is.null(o$full_fit)) length(o$full_fit$sets$cs) else NA_integer_,
         ser_fallback = o$ser_fallback %||% NA, Q_art = o$Q_art %||% NA_real_,
         DCS_max = if (is.null(o$cs_max)) NA_real_ else max(c(0, o$cs_max), na.rm = TRUE),
         qart_matches_fit = o$qart_matches_fit %||% NA, time_s = round(o$time_s, 1))
})
chk %>% arrange(unit, factor(method, levels = c("insample", "susie_rss", "susie_rrss"))) %>% as.data.frame()

## refit class vs the class from the stored coloc diagnostics
chk %>% filter(method == "susie_rrss") %>%
  mutate(trigger_refit = case_when(Q_art > 0.1 & DCS_max >= 30 ~ "Both", Q_art > 0.1 ~ "Q_art only",
                                   DCS_max >= 30 ~ "D_CS only", TRUE ~ "none")) %>%
  right_join(select(cases, unit, trigger, Q_art_stored = Q_art, DCS_max_stored = DCS_max), by = "unit") %>%
  select(unit, trigger, trigger_refit, Q_art_stored, Q_art, DCS_max_stored, DCS_max) %>%
  arrange(trigger) %>% as.data.frame()

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


ERROR: [1m[33mError[39m in `arrange()`:[22m
[1m[22m[36mℹ[39m In argument: `..1 = unit`.
[1mCaused by error:[22m
[1m[22m[33m![39m `..1` must be a vector, not a function.
